# Task 5: Strategy Backtesting
## Comparing Optimal Portfolio vs Benchmark Strategy

This notebook implements:
1. Backtesting window isolation (final year held out from training)
2. Benchmark portfolio (60% SPY / 40% BND)
3. Strategy simulation using optimal weights
4. Cumulative returns comparison
5. Performance metrics calculation

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## 1. Load Data and Setup Backtesting Window

In [ ]:
# Load all required data
prices = pd.read_csv('../data/raw/stock_prices.csv', index_col=0, parse_dates=True)
returns = pd.read_csv('../data/processed/daily_returns.csv', index_col=0, parse_dates=True)
optimal_weights = pd.read_csv('../data/processed/optimal_weights.csv')

TICKERS = ['TSLA', 'BND', 'SPY']
INITIAL_CAPITAL = 100000
RISK_FREE_RATE = 0.02

print("Data loaded successfully!")
print(f"\nPrice data shape: {prices.shape}")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")

print("\n=== Optimal Weights from Task 4 ===")
display(optimal_weights)

## 2. Backtesting Window: Final Year Isolation

The final year of data is held out for backtesting (not used in model training).

In [ ]:
# Define backtesting period: final year of data
BACKTEST_DAYS = 252  # ~1 year of trading days

# Isolate backtesting period
total_days = len(prices)
backtest_start_idx = total_days - BACKTEST_DAYS

backtest_prices = prices.iloc[backtest_start_idx:].copy()
backtest_returns = returns.iloc[backtest_start_idx:].copy()

print("="*60)
print("BACKTESTING WINDOW SETUP")
print("="*60)
print(f"\nTotal historical data: {total_days} trading days")
print(f"Training period: {prices.index[0].date()} to {prices.index[backtest_start_idx-1].date()}")
print(f"   ({backtest_start_idx} days - used for model training)")
print(f"\nBacktesting period: {backtest_prices.index[0].date()} to {backtest_prices.index[-1].date()}")
print(f"   ({len(backtest_prices)} days - held out for testing)")

## 3. Define Benchmark Strategy

Benchmark: 60% SPY / 40% BND (classic balanced portfolio)

In [ ]:
# Define benchmark weights
benchmark_weights = {
    'TSLA': 0.00,   # No TSLA in benchmark
    'BND': 0.40,    # 40% bonds
    'SPY': 0.60     # 60% equities
}

print("=== Benchmark Portfolio ===")
print("\nBenchmark: 60% SPY / 40% BND")
print("(Classic balanced portfolio - no TSLA exposure)")

# Define optimal portfolio weights from Task 4
max_sharpe_weights_dict = {
    'TSLA': optimal_weights.loc[optimal_weights['Ticker'] == 'TSLA', 'Max_Sharpe_Weight'].values[0],
    'BND': optimal_weights.loc[optimal_weights['Ticker'] == 'BND', 'Max_Sharpe_Weight'].values[0],
    'SPY': optimal_weights.loc[optimal_weights['Ticker'] == 'SPY', 'Max_Sharpe_Weight'].values[0]
}

min_vol_weights_dict = {
    'TSLA': optimal_weights.loc[optimal_weights['Ticker'] == 'TSLA', 'Min_Vol_Weight'].values[0],
    'BND': optimal_weights.loc[optimal_weights['Ticker'] == 'BND', 'Min_Vol_Weight'].values[0],
    'SPY': optimal_weights.loc[optimal_weights['Ticker'] == 'SPY', 'Min_Vol_Weight'].values[0]
}

print(f"\n=== Max Sharpe Portfolio ===")
for ticker, weight in max_sharpe_weights_dict.items():
    print(f"  {ticker}: {weight:.2%}")

print(f"\n=== Min Volatility Portfolio ===")
for ticker, weight in min_vol_weights_dict.items():
    print(f"  {ticker}: {weight:.2%}")

## 4. Strategy Simulation Functions

In [ ]:
def simulate_portfolio(prices, weights_dict, initial_capital=INITIAL_CAPITAL, 
                      rebalance_freq='M'):
    """
    Simulate portfolio performance with given weights.
    
    Parameters:
    - prices: DataFrame of historical prices
    - weights_dict: Dictionary of target weights
    - initial_capital: Starting portfolio value
    - rebalance_freq: 'M' for monthly, 'Q' for quarterly, None for static
    
    Returns:
    - DataFrame with portfolio values over time
    - Dictionary of performance metrics
    """
    # Get returns
    returns = prices.pct_change().dropna()
    
    # Calculate daily portfolio returns
    portfolio_returns = sum(returns[ticker] * weight for ticker, weight in weights_dict.items())
    
    # Calculate cumulative returns
    cumulative = (1 + portfolio_returns).cumprod()
    portfolio_values = cumulative * initial_capital
    
    return portfolio_values, portfolio_returns

In [ ]:
# Run simulations
print("\n=== Running Portfolio Simulations ===")

# Benchmark simulation
benchmark_values, benchmark_rets = simulate_portfolio(backtest_prices, benchmark_weights)

# Max Sharpe strategy simulation
max_sharpe_values, max_sharpe_rets = simulate_portfolio(backtest_prices, max_sharpe_weights_dict)

# Min Volatility strategy simulation
min_vol_values, min_vol_rets = simulate_portfolio(backtest_prices, min_vol_weights_dict)

print("\nSimulations complete!")
print(f"Benchmark final value: ${benchmark_values.iloc[-1]:,.2f}")
print(f"Max Sharpe final value: ${max_sharpe_values.iloc[-1]:,.2f}")
print(f"Min Vol final value: ${min_vol_values.iloc[-1]:,.2f}")

## 5. Cumulative Returns Comparison

In [ ]:
# Create cumulative returns DataFrame
cumulative_returns = pd.DataFrame({
    'Benchmark (60/40)': (1 + benchmark_rets).cumprod(),
    'Max Sharpe Strategy': (1 + max_sharpe_rets).cumprod(),
    'Min Volatility Strategy': (1 + min_vol_rets).cumprod()
})

# Visualization
fig, ax = plt.subplots(figsize=(14, 8))

cumulative_returns.plot(ax=ax, linewidth=2)

ax.set_title('Cumulative Returns: Strategy vs Benchmark', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return (Starting = 1.0)')
ax.axhline(y=1, color='black', linestyle='--', alpha=0.5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Add annotations for final values
for col in cumulative_returns.columns:
    final_val = cumulative_returns[col].iloc[-1]
    ax.annotate(f'{final_val:.2f}', 
                xy=(cumulative_returns.index[-1], final_val),
                xytext=(10, 0), textcoords='offset points')

plt.tight_layout()
plt.savefig('../data/processed/cumulative_returns_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/cumulative_returns_comparison.png")

In [ ]:
# Portfolio values over time
fig, ax = plt.subplots(figsize=(14, 8))

ax.plot(benchmark_values.index, benchmark_values, label='Benchmark (60/40)', linewidth=2)
ax.plot(max_sharpe_values.index, max_sharpe_values, label='Max Sharpe Strategy', linewidth=2)
ax.plot(min_vol_values.index, min_vol_values, label='Min Volatility Strategy', linewidth=2)

ax.axhline(y=INITIAL_CAPITAL, color='black', linestyle='--', alpha=0.5, label='Initial Capital')

ax.set_title('Portfolio Values Over Time: Backtesting Period', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Portfolio Value ($)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Format y-axis as currency
from matplotlib.ticker import FuncFormatter
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../data/processed/portfolio_values_backtest.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/portfolio_values_backtest.png")

## 6. Performance Metrics Calculation

In [ ]:
def calculate_performance_metrics(returns, risk_free_rate=RISK_FREE_RATE):
    """Calculate comprehensive performance metrics."""
    
    # Total return
    cumulative_return = (1 + returns).prod() - 1
    
    # Annualized return
    trading_days = len(returns)
    annualized_return = returns.mean() * 252
    
    # Annualized volatility
    annualized_vol = returns.std() * np.sqrt(252)
    
    # Sharpe Ratio
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol > 0 else 0
    
    # Maximum drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Calmar Ratio
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else np.inf
    
    # Sortino Ratio (downside deviation)
    downside_std = returns[returns < 0].std() * np.sqrt(252) if len(returns[returns < 0]) > 0 else 0.001
    sortino_ratio = (annualized_return - risk_free_rate) / downside_std
    
    # Win rate
    win_rate = (returns > 0).sum() / len(returns)
    
    return {
        'Total Return': cumulative_return,
        'Annualized Return': annualized_return,
        'Annualized Volatility': annualized_vol,
        'Sharpe Ratio': sharpe_ratio,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': calmar_ratio,
        'Sortino Ratio': sortino_ratio,
        'Win Rate': win_rate
    }

In [ ]:
# Calculate metrics for all strategies
print("="*80)
print("PERFORMANCE METRICS COMPARISON")
print("="*80)

strategies = {
    'Benchmark (60/40)': benchmark_rets,
    'Max Sharpe Strategy': max_sharpe_rets,
    'Min Volatility Strategy': min_vol_rets
}

metrics_comparison = {}

for name, rets in strategies.items():
    metrics = calculate_performance_metrics(rets)
    metrics_comparison[name] = metrics
    
    print(f"\n### {name} ###")
    print(f"  Total Return: {metrics['Total Return']:.2%}")
    print(f"  Annualized Return: {metrics['Annualized Return']:.2%}")
    print(f"  Annualized Volatility: {metrics['Annualized Volatility']:.2%}")
    print(f"  Sharpe Ratio: {metrics['Sharpe Ratio']:.4f}")
    print(f"  Max Drawdown: {metrics['Max Drawdown']:.2%}")
    print(f"  Sortino Ratio: {metrics['Sortino Ratio']:.4f}")
    print(f"  Win Rate: {metrics['Win Rate']:.2%}")

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame(metrics_comparison).T

# Format for display
display_df = comparison_df.copy()
for col in ['Total Return', 'Annualized Return', 'Annualized Volatility', 'Max Drawdown', 'Win Rate']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.2%}")
for col in ['Sharpe Ratio', 'Calmar Ratio', 'Sortino Ratio']:
    display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")

print("\n=== Performance Metrics Summary ===")
display(display_df)

# Save results
comparison_df.to_csv('../data/processed/backtest_performance_metrics.csv')
print("\nResults saved to data/processed/backtest_performance_metrics.csv")

## 7. Drawdown Analysis

In [ ]:
# Drawdown visualization
fig, ax = plt.subplots(figsize=(14, 6))

for name, rets in strategies.items():
    cumulative = (1 + rets).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    drawdown.plot(ax=ax, label=name, linewidth=1.5)

ax.set_title('Portfolio Drawdown Comparison', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.legend()
ax.grid(True, alpha=0.3)

# Format as percentage
from matplotlib.ticker import FuncFormatter
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('../data/processed/drawdown_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved to data/processed/drawdown_comparison.png")

## 8. Summary and Recommendations

In [ ]:
# Determine best performing strategy
print("="*80)
print("BACKTESTING SUMMARY")
print("="*80)

# Best by Sharpe
best_sharpe = comparison_df['Sharpe Ratio'].idxmax()
print(f"\nBest Risk-Adjusted Return (Sharpe): {best_sharpe}")
print(f"  Sharpe Ratio: {comparison_df.loc[best_sharpe, 'Sharpe Ratio']:.4f}")

# Best by Total Return
best_return = comparison_df['Total Return'].idxmax()
print(f"\nBest Total Return: {best_return}")
print(f"  Total Return: {comparison_df.loc[best_return, 'Total Return']:.2%}")

# Lowest Volatility
lowest_vol = comparison_df['Annualized Volatility'].idxmin()
print(f"\nLowest Volatility: {lowest_vol}")
print(f"  Annualized Volatility: {comparison_df.loc[lowest_vol, 'Annualized Volatility']:.2%}")

# vs Benchmark comparison
print(f"\n=== Max Sharpe Strategy vs Benchmark ===")
sharpe_vs_bench = comparison_df.loc['Max Sharpe Strategy', 'Sharpe Ratio'] - comparison_df.loc['Benchmark (60/40)', 'Sharpe Ratio']
ret_vs_bench = comparison_df.loc['Max Sharpe Strategy', 'Total Return'] - comparison_df.loc['Benchmark (60/40)', 'Total Return']
print(f"Sharpe Difference: {sharpe_vs_bench:+.4f}")
print(f"Return Difference: {ret_vs_bench:+.2%}")

if sharpe_vs_bench > 0:
    print("\n*** RECOMMENDATION: Max Sharpe Strategy outperforms Benchmark on risk-adjusted basis ***")
else:
    print("\n*** RECOMMENDATION: Benchmark performs better on risk-adjusted basis ***")

In [ ]:
# Save final backtest results
final_results = pd.DataFrame({
    'Strategy': ['Benchmark (60/40)', 'Max Sharpe', 'Min Volatility'],
    'Initial Capital': [INITIAL_CAPITAL, INITIAL_CAPITAL, INITIAL_CAPITAL],
    'Final Value': [benchmark_values.iloc[-1], max_sharpe_values.iloc[-1], min_vol_values.iloc[-1]],
    'Total Return': [comparison_df.loc['Benchmark (60/40)', 'Total Return'],
                     comparison_df.loc['Max Sharpe Strategy', 'Total Return'],
                     comparison_df.loc['Min Volatility Strategy', 'Total Return']],
    'Sharpe Ratio': [comparison_df.loc['Benchmark (60/40)', 'Sharpe Ratio'],
                     comparison_df.loc['Max Sharpe Strategy', 'Sharpe Ratio'],
                     comparison_df.loc['Min Volatility Strategy', 'Sharpe Ratio']],
    'Max Drawdown': [comparison_df.loc['Benchmark (60/40)', 'Max Drawdown'],
                      comparison_df.loc['Max Sharpe Strategy', 'Max Drawdown'],
                      comparison_df.loc['Min Volatility Strategy', 'Max Drawdown']]
})

final_results.to_csv('../data/processed/backtest_final_results.csv', index=False)

print("\nFinal backtest results saved.")
display(final_results)

## Summary

### Key Findings:

1. **Backtesting Period**: Final year isolated from training data
   - Ensures realistic out-of-sample performance evaluation

2. **Benchmark**: Classic 60% SPY / 40% BND portfolio
   - Represents traditional balanced investment approach
   - No exposure to TSLA

3. **Optimal Strategy Performance**:
   - Max Sharpe Strategy incorporates TSLA forecast
   - Compared against benchmark on multiple metrics

4. **Performance Metrics**:
   - Total Return, Annualized Return, Sharpe Ratio
   - Maximum Drawdown, Calmar Ratio, Sortino Ratio

### All Tasks Complete:
- Task 1: Data extraction, cleaning, EDA ✓
- Task 2: ARIMA/SARIMA/LSTM models ✓
- Task 3: Future forecasting ✓
- Task 4: Portfolio optimization ✓
- Task 5: Backtesting ✓